In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 4


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 5.336466647684574
Epoch 2/100, Loss: 4.427559971809387
Epoch 3/100, Loss: 4.887803874909878
Epoch 4/100, Loss: 4.232090674340725
Epoch 5/100, Loss: 4.701523512601852
Epoch 6/100, Loss: 4.699500605463982
Epoch 7/100, Loss: 5.182018645107746
Epoch 8/100, Loss: 5.3094410970807076
Epoch 9/100, Loss: 5.066309295594692
Epoch 10/100, Loss: 5.653543680906296
Epoch 11/100, Loss: 4.553400136530399
Epoch 12/100, Loss: 4.745679385960102
Epoch 13/100, Loss: 4.7852074801921844
Epoch 14/100, Loss: 4.937862932682037
Epoch 15/100, Loss: 4.43836784362793
Epoch 16/100, Loss: 5.0347418785095215


Epoch 17/100, Loss: 5.021825611591339
Epoch 18/100, Loss: 4.72497995942831
Epoch 19/100, Loss: 4.902732104063034
Epoch 20/100, Loss: 4.510999973863363
Epoch 21/100, Loss: 4.503953341394663
Epoch 22/100, Loss: 6.2422881200909615
Epoch 23/100, Loss: 4.6992509588599205
Epoch 24/100, Loss: 4.318152032792568
Epoch 25/100, Loss: 4.90099635720253
Epoch 26/100, Loss: 4.841448109596968
Epoch 27/100, Loss: 4.425222299993038
Epoch 28/100, Loss: 4.614585906267166
Epoch 29/100, Loss: 4.707480914890766
Epoch 30/100, Loss: 4.232269994914532
Epoch 31/100, Loss: 4.554698757827282
Epoch 32/100, Loss: 4.79944459348917


Epoch 33/100, Loss: 4.901455968618393
Epoch 34/100, Loss: 4.492105171084404
Epoch 35/100, Loss: 4.150566920638084
Epoch 36/100, Loss: 5.0284358859062195
Epoch 37/100, Loss: 4.624787986278534
Epoch 38/100, Loss: 3.832557510584593
Epoch 39/100, Loss: 5.079403266310692
Epoch 40/100, Loss: 5.007844872772694
Epoch 41/100, Loss: 4.932151526212692
Epoch 42/100, Loss: 4.339665375649929
Epoch 43/100, Loss: 5.22596737742424
Epoch 44/100, Loss: 4.500931583344936
Epoch 45/100, Loss: 4.938420355319977
Epoch 46/100, Loss: 3.97709821164608
Epoch 47/100, Loss: 4.473969668149948
Epoch 48/100, Loss: 3.9296439588069916


Epoch 49/100, Loss: 4.1910750940442085
Epoch 50/100, Loss: 4.00892511382699
Epoch 51/100, Loss: 3.757429152727127
Epoch 52/100, Loss: 4.987947799265385
Epoch 53/100, Loss: 4.66007287427783
Epoch 54/100, Loss: 4.272987544536591
Epoch 55/100, Loss: 5.039771109819412
Epoch 56/100, Loss: 4.475351549685001
Epoch 57/100, Loss: 4.637935105711222
Epoch 58/100, Loss: 5.004638805985451
Epoch 59/100, Loss: 4.631642863154411
Epoch 60/100, Loss: 4.972450755536556
Epoch 61/100, Loss: 4.519320346415043
Epoch 62/100, Loss: 3.879102848470211


Epoch 63/100, Loss: 4.630123391747475
Epoch 64/100, Loss: 4.943734481930733
Epoch 65/100, Loss: 4.519789393991232
Epoch 66/100, Loss: 3.793685205280781
Epoch 67/100, Loss: 4.333407573401928
Epoch 68/100, Loss: 4.583724834024906
Epoch 69/100, Loss: 4.311988525092602
Epoch 70/100, Loss: 4.558416172862053
Epoch 71/100, Loss: 4.673612654209137
Epoch 72/100, Loss: 4.355059280991554
Epoch 73/100, Loss: 4.450515426695347
Epoch 74/100, Loss: 4.486964210867882
Epoch 75/100, Loss: 5.000582858920097
Epoch 76/100, Loss: 5.0884543135762215
Epoch 77/100, Loss: 5.074480377137661
Epoch 78/100, Loss: 5.17362717539072


...

Epoch 31/100, Loss: 1.6101294197142124
Epoch 32/100, Loss: 1.5985709354281425
Epoch 33/100, Loss: 1.735482957214117
Epoch 34/100, Loss: 1.6108217984437943
Epoch 35/100, Loss: 1.715334925800562
Epoch 36/100, Loss: 1.4584328010678291
Epoch 37/100, Loss: 1.6513813063502312
Epoch 38/100, Loss: 1.5908118337392807
Epoch 39/100, Loss: 1.646794132888317
Epoch 40/100, Loss: 1.6363883577287197
Epoch 41/100, Loss: 1.57907073199749
Epoch 42/100, Loss: 1.611301340162754
Epoch 43/100, Loss: 1.685856193304062
Epoch 44/100, Loss: 1.5988257564604282
Epoch 45/100, Loss: 1.590913712978363


Epoch 46/100, Loss: 1.7580182403326035
Epoch 47/100, Loss: 1.6120598763227463
Epoch 48/100, Loss: 1.440895576030016
Epoch 49/100, Loss: 1.532054789364338
Epoch 50/100, Loss: 1.6232518218457699
Epoch 51/100, Loss: 1.6230058073997498
Epoch 52/100, Loss: 1.5580409578979015
Epoch 53/100, Loss: 1.6027279198169708
Epoch 54/100, Loss: 1.655973207205534
Epoch 55/100, Loss: 1.6179752871394157
Epoch 56/100, Loss: 1.680672101676464
Epoch 57/100, Loss: 1.6919737979769707
Epoch 58/100, Loss: 2.0763083770871162
Epoch 59/100, Loss: 1.6952888444066048
Epoch 60/100, Loss: 1.5411339476704597
Epoch 61/100, Loss: 1.7106698900461197


Epoch 62/100, Loss: 1.7895470783114433
Epoch 63/100, Loss: 1.5543040856719017
Epoch 64/100, Loss: 1.7117766924202442
Epoch 65/100, Loss: 1.7357519194483757
Epoch 66/100, Loss: 1.5932913571596146
Epoch 67/100, Loss: 1.6505579352378845
Epoch 68/100, Loss: 1.7150020487606525
Epoch 69/100, Loss: 1.7231722995638847
Epoch 70/100, Loss: 1.7225563079118729
Epoch 71/100, Loss: 1.6455223709344864
Epoch 72/100, Loss: 1.5787992179393768
Epoch 73/100, Loss: 1.5915232971310616
Epoch 74/100, Loss: 1.54524327814579


Epoch 75/100, Loss: 1.5454238019883633
Epoch 76/100, Loss: 1.639875888824463
Epoch 77/100, Loss: 1.5850922651588917
Epoch 78/100, Loss: 1.5263638868927956
Epoch 79/100, Loss: 1.7866041027009487
Epoch 80/100, Loss: 1.6994411163032055
Epoch 81/100, Loss: 1.6751151606440544
Epoch 82/100, Loss: 1.727410763502121
Epoch 83/100, Loss: 1.5852045342326164
Epoch 84/100, Loss: 1.6899943947792053
Epoch 85/100, Loss: 1.7047208473086357
Epoch 86/100, Loss: 1.5742567032575607
Epoch 87/100, Loss: 1.8286241069436073
Epoch 88/100, Loss: 1.7142039313912392
Epoch 89/100, Loss: 1.6268597096204758
Epoch 90/100, Loss: 1.6584280952811241
Epoch 91/100, Loss: 1.6351808793842793


Epoch 92/100, Loss: 1.7221543081104755
Epoch 93/100, Loss: 1.6873083114624023
Epoch 94/100, Loss: 1.4756432324647903
Epoch 95/100, Loss: 1.6183889545500278
Epoch 96/100, Loss: 1.6305514983832836
Epoch 97/100, Loss: 1.6693402230739594
Epoch 98/100, Loss: 1.52052753418684
Epoch 99/100, Loss: 1.5582422316074371
Epoch 100/100, Loss: 1.5941206514835358
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5516
